# 华东杯数学建模 — 量化因子研究与优化 工作汇总## 概述本笔记本汇总了在量化因子研究中的所有工作成果，包括：1. **8大优化方法** (A-H类)：时序排序、滚动Z-Score、波动率缩放、衰减线性加权、正交残差、因子动量Delta、有符号幂变换、交叉共振2. **有符号幂变换 (Signed Power Transformation)** 的深入研究（Round 1 & Round 2）3. **高阶因子构建**：分数阶差分 (Fractional Differentiation)、卡尔曼滤波 (Kalman Filter)、成交量分布POC距离4. **文献综述因子**：基于学术文献的12个经典因子实现与评估所有因子的IC表现已汇总至 `Consolidated_Factor_IC_Summary.csv`。

---# 第一部分：因子类型与优化方法总览## 1.1 基础优化方法分类 (A-H 八大类)### A. 时序排序 (Time-Series Ranking)将原始信号在其历史窗口内进行分位数排名，将不同量纲的信号归一化到[0,1]区间。$$\text{TsRank}(x, w)_t = \frac{1}{w}\sum_{i=t-w+1}^{t} \mathbb{1}[x_i \leq x_t]$$- **A1_TsRank5_GK_Vol**: GK波动率的5期时序排名 → 识别波动率扩张的相对位置- **A2_TsRank20_OrderImbalance**: 订单不平衡的20期时序排名 → 捕捉买卖压力的极端状态- **A3_TsRank60_LogHL_Range**: 对数高低价差的60期排名 → 波幅的相对大小- **A4_TsRank10_VWAP_Dev**: VWAP偏离的10期排名 → 价格相对均价偏离的极值识别### B. 滚动Z-Score (Rolling Z-Score)在滚动窗口内将信号标准化为标准正态分布，衡量当前值偏离均值的标准差倍数。$$\text{ZScore}(x, w)_t = \frac{x_t - \mu_w(x_t)}{\sigma_w(x_t) + \epsilon}$$- **B1_ZScore60_GK_Vol**: GK波动率Z-Score → 波动率异常程度- **B2_ZScore60_OrderImbalance**: 订单不平衡Z-Score → 买卖失衡的统计显著性- **B3_ZScore60_LogHL_Range**: 高低价差Z-Score → 日内波幅异常检测- **B4_ZScore30_Close_Position**: 收盘位置Z-Score → 尾盘效应显著性### C. 波动率缩放 (Volatility Scaling)用滚动波动率对信号进行缩放，消除异方差性，使信号在不同波动率环境下可比。$$\text{VolScaled}(x)_t = \frac{x_t}{\sqrt{\text{MA}(\text{GKVol}, w)_t} + \epsilon}$$- **C1_VolScaled_ROC_5M**: 波动率缩放5期变化率- **C2_VolScaled_ROC_60M**: 波动率缩放60期变化率- **C3_VolScaled_MACD_60M**: 波动率缩放MACD- **C4_VolScaled_OrderImbalance**: 波动率缩放订单不平衡### D. 衰减线性加权 (Decay Linear Weighting)对信号进行时间衰减加权平均，近期值权重更大，捕捉信号的短期趋势。$$\text{DecayLinear}(x, w)_t = \sum_{i=0}^{w-1} \frac{w-i}{\sum_{j=1}^{w} j} \cdot x_{t-i}$$- **D1_DecayLinear5_Returns**: 5期衰减收益率- **D2_DecayLinear10_GK_Vol**: 10期衰减GK波动率- **D3_DecayLinear20_LogHL**: 20期衰减对数高低价差### E. 正交残差 (Orthogonal Residuals)通过滚动回归去除两个信号间的共线性，提取独立于第二信号的残差分量。$$\text{Orth}(x, y, w)_t = x_t - \beta_t \cdot y_t, \quad \beta_t = \frac{\text{Cov}_w(x,y)}{\text{Var}_w(y) + \epsilon}$$- **E1_Orth_EMA_Ret_vs_GKVol**: EMA收益正交于波动率 → 纯Alpha（剔除波动率影响）- **E2_Orth_VWAP_Dev_vs_LogHL**: VWAP偏离正交于波幅 → 纯价格偏离- **E3_Orth_OrderImb_vs_GKVol**: 订单不平衡正交于波动率 → 纯买卖压力### F. 因子动量Delta (Factor Momentum Delta)对因子本身计算差分，捕捉因子方向的变化趋势（二阶信号）。$$\text{Delta}(x, d)_t = x_t - x_{t-d}$$- **F1_Delta3_OrderImbalance**: 3期订单不平衡变化- **F2_Delta5_GK_Vol**: 5期波动率变化- **F3_Delta5_LogHL_Range**: 5期波幅变化### G. 有符号幂变换 (Signed Power Transformation) ★核心方法详见第二部分专题介绍。### H. 交叉共振 (Cross Resonance)将两个互补信号相乘，捕捉联合分布中的非线性交互效应。$$\text{Cross}(x, y)_t = x_t \cdot y_t$$- **H1_OI_x_GKVol**: 订单不平衡 × 波动率 → 放量失衡- **H2_VWAP_Dev_x_LogHL**: VWAP偏离 × 波幅 → 大波动偏离

---# 第二部分：有符号幂变换 (Signed Power Transformation) 专题## 2.1 方法简介有符号幂变换是本研究中表现最为突出的因子优化方法。其核心思想是：> **对信号的幅值进行非线性缩放，同时完整保留信号的符号（方向性信息）。**## 2.2 数学定义### 核心公式对于方向性信号（可正可负），使用 **Signed Power**：$$\text{SignedPower}(x, p) = \text{sign}(x) \cdot |x|^p$$对于非负信号（波动率、价差等），使用 **Power Only**：$$\text{PowerOnly}(x, p) = \max(x, 0)^p$$### 参数含义| 参数 p | 效果 | 适用场景 | 典型信号 ||--------|------|---------|---------|| $p > 1$ (如1.5, 2.0) | **幅值放大** — 强化极端信号，弱化微弱噪声 | 方向性信号：收益、偏离、动量、震荡指标 | ema_ret, roc, vwap_dev, mom || $p = 1$ | 恒等变换 — 保持原始线性关系 | — | — || $p < 1$ (如0.5, 0.3) | **右尾压缩** — 抑制极端值，缩小量纲差异 | 正值信号：波动率、价差、流动性指标 | gk_vol, hl_range, natr, amihud |## 2.3 为什么有符号幂变换有效？### 1. 非线性信息提取金融市场中，信号强度与未来收益的关系往往是非线性的。例如：- 微小的价格偏离（|VWAP偏离| < 0.1%）几乎不包含预测信息（噪声）- 但较大的偏离（|VWAP偏离| > 1%）往往预示着强烈的均值回归$p > 1$ 的幂变换**放大强信号、压缩弱噪声**，自动实现了"信号-噪声分离"。### 2. 方向性保持与传统的 $x^2$ 或 $|x|^p$ 不同，`SignedPower` 通过 $\text{sign}(x)$ 保留原始方向：- 正信号 → 正因子值（看多特征）- 负信号 → 负因子值（看空特征）这确保了因子与收益的IC关系不发生方向反转。### 3. 尾部行为调节- **p > 1**: 对极端值更敏感 → 适合"极端信号预测极端收益"的策略- **0 < p < 1**: 对极端值不敏感 → 适合"稳健信号稳定预测"的策略## 2.4 实验设计### Round 1: 四大幂次方案对比| 方案 | 幂次 | 策略 | 信号数量 | 代表因子 ||------|------|------|---------|---------|| SP15 | p=1.5 | 温和放大（方向性信号）+ 压缩（波动率信号） | 14方向 + 6压缩 | SP15_ema_ret_60M, SP15_roc_5M, SP05_GK_Vol || SP20 | p=2.0 | 强放大（方向性信号）+ 凸变换（有界信号） | 8方向 + 2有界 | SP20_ema_ret_60M, SP20_roc_5M, SP20_Close_Position || SP05 | p=0.5 | 平方根压缩（波动率/价差信号） | 6正值 | SP05_Log_HL_Range, SP05_GK_Volatility_1M || SP03 | p=0.3 | 强压缩（尾部极度压缩） | 5正值 | SP03_Log_HL_Range, SP03_GK_Volatility_1M |### Round 2: 新基础信号的幂变换在Round 1基础上引入18个新的基础信号（HMA偏离、KAMA偏离、一目均衡溢价、CMO、Aroon等），系统性地比较 p=1.5 和 p=2.0 在同一原始信号上的效果。## 2.5 IC表现汇总从实验结果来看，**p=2.0 (平方变换)** 的综合性得分普遍最高，其次是 **p=1.5**。两类放大型幂变换的效果远优于压缩型（p<1），表明在该数据集中**强化极端信号**比抑制极端值更有效。Top Signed Power因子（按Comprehensive Score排序）：| 排名 | 因子 | 幂方案 | Ret5 IC Mean | Ret60 IC Mean | Comp Score ||------|------|--------|-------------|--------------|------------|| 1 | SP20_ema_ret_60M | SP20 p=2.0 | -0.0327 | -0.0466 | 0.1359 || 2 | SP20_ema_ret_15M | SP20 p=2.0 | -0.0354 | -0.0376 | 0.1138 || 3 | SP15_roc_5M | SP15 p=1.5 | -0.0349 | -0.0172 | 0.1080 || 4 | SP15_ema_ret_60M | SP15 p=1.5 | -0.0268 | -0.0395 | 0.1073 || 5 | SP15_ema_ret_15M | SP15 p=1.5 | -0.0326 | -0.0344 | 0.1036 |> **关键发现**：EMA收益偏离（ema_ret）经过Signed Power变换后成为表现最稳定的因子类型，> 无论是在短期(Ret5)还是长期(Ret60)预测窗口上。

---# 第三部分：因子目录 — 名称、公式与标的特征意义## 3.1 基础因子的原始信号以下基础信号是所有优化因子的构建原材料：| 信号名称 | 计算公式 | 经济意义 ||---------|---------|---------|| **log_hl** | $\ln(\max(H, L+\epsilon) / \max(L, \epsilon))$ | 对数高低价差 — 日内价格波动幅度 || **log_oc** | $\ln(\max(O, C+\epsilon) / \max(C, \epsilon))$ | 对数开收盘价差 — 日内方向性变动 || **gk_vol** | $\max(0.5 \cdot \log\_hl^2 - (2\ln 2 - 1) \cdot \log\_oc^2, 0)$ | Parkinson-Garman-Klass 波动率估计量 — 比收盘价-收盘价波动率高5倍效率 || **log_hl_range** | $\ln(H/L)$ | 对数波幅 — 价格极端波动范围 || **close_position** | $(C-L)/(H-L)$ | 收盘位置 — 多方/空方控盘程度，接近1=买方主导，接近0=卖方主导 || **log_ret** | $\ln(C_t / C_{t-1})$ | 对数收益率 — 价格变动方向和幅度 || **order_imbalance** | $(C_t - C_{t-1}) / (\ln(\max(V_t, 1)) + \epsilon)$ | 订单不平衡代理 — 每单位对数成交量的价格变动，捕捉买卖压力 || **vwap_dev** | $(C - \text{OHLC4}) / \text{OHLC4}$ | VWAP偏离 — 收盘价与OHLC均价的偏差，反映日内价格趋势强弱 || **roc_5 / roc_60** | $(C_t - C_{t-5})/C_{t-5}$ 或 $(C_t - C_{t-60})/C_{t-60}$ | 价格变化率 (Rate of Change) — 动量的简单度量 || **macd_line** | $\text{EMA}_{12}(C) - \text{EMA}_{26}(C)$ | MACD线 — 短期与长期趋势的差值，捕捉趋势切换 || **ema_ret** | $(C - \text{EMA}_{60}(C)) / \text{EMA}_{60}(C)$ | EMA收益偏离 — 价格相对于60期指数均线的偏离程度 |

## 3.2 A-H 优化因子详细目录### A. 时序排序因子 (Time-Series Ranking)**原理**：在历史窗口内将原始信号转化为分位数排名[0,1]，消除量纲差异，使不同信号可比较。| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **A1_TsRank5_GK_Vol** | $\text{TsRank}(\text{gk\_vol}, 5)$ | **短期波动率排序** — 当前波动率在近5期的分位数。高值=波动放量中，低值=波动缩量中。用于识别波动率扩张/收缩的相对极端状态 || **A2_TsRank20_OrderImbalance** | $\text{TsRank}(\text{order\_imbalance}, 20)$ | **中期买卖压力排序** — 当前订单不平衡在近20期的排名。高值=买方压力极端，低值=卖方压力极端。可用于反向交易（极端压力后反转） || **A3_TsRank60_LogHL_Range** | $\text{TsRank}(\text{log\_hl\_range}, 60)$ | **长期波幅排序** — 当前日内波幅在60期的分位数。高=大波动日，低=窄幅震荡日。用于识别波动率区制切换 || **A4_TsRank10_VWAP_Dev** | $\text{TsRank}(\text{vwap\_dev}, 10)$ | **短期均价偏离排序** — VWAP偏离在近10期的分位数。极端排名预示均值回归机会 |### B. 滚动Z-Score因子**原理**：信号减去滚动均值后除以滚动标准差，度量当前值相对于历史的"异常程度"。| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **B1_ZScore60_GK_Vol** | $\text{ZScore}(\text{gk\_vol}, 60)$ | **波动率异常** — 当前GK波动率偏离60期均值的标准差数。|Z|>2=异常波动，预示波动率均值回归 || **B2_ZScore60_OrderImbalance** | $\text{ZScore}(\text{order\_imbalance}, 60)$ | **买卖失衡异常** — 当前订单不平衡的统计显著性。显著正值=超买信号，显著负值=超卖信号 || **B3_ZScore60_LogHL_Range** | $\text{ZScore}(\text{log\_hl\_range}, 60)$ | **波幅异常** — 日内振幅的统计偏离。识别突破性大阳/大阴线与普通K线的差异 || **B4_ZScore30_Close_Position** | $\text{ZScore}(\text{close\_position}, 30)$ | **尾盘效应异常** — 收盘位置的统计偏离。高Z=异常强势收盘（尾盘拉抬），低Z=异常弱势收盘（尾盘砸盘） |### C. 波动率缩放因子**原理**：将趋势/动量信号除以滚动波动率，消除"高波动期信号天然大"的偏差。| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **C1_VolScaled_ROC_5M** | $\text{roc\_5} / \sqrt{\text{MA}(\text{gk\_vol}, 5)}$ | **波动率调整短期动量** — 剔除波动率影响后的5期收益率。高波动下的5%涨幅可能不如低波动下的2%涨幅有意义 || **C2_VolScaled_ROC_60M** | $\text{roc\_60} / \sqrt{\text{MA}(\text{gk\_vol}, 60)}$ | **波动率调整中长期动量** — 经风险调整的60期趋势强度 || **C3_VolScaled_MACD_60M** | $\text{macd} / \sqrt{\text{MA}(\text{gk\_vol}, 60)}$ | **波动率调整趋势切换** — 标准化后的MACD，区分"真正的趋势转变"与"高波动噪声" || **C4_VolScaled_OrderImbalance** | $\text{order\_imbalance} / \sqrt{\text{MA}(\text{gk\_vol}, 20)}$ | **波动率调整买卖压力** — 标准化订单不平衡，分辨"真实买压"与"波动率驱动的伪买压" |### D. 衰减线性加权因子**原理**：近期值权重大、远期值权重小，比简单移动平均更灵敏地捕捉趋势转变。| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **D1_DecayLinear5_Returns** | $\sum_{i=0}^{4} w_i \cdot r_{t-i}, w_i = \frac{5-i}{15}$ | **衰减加权利率** — 最近5期收益的加权平均。比SMA更快、比单期收益更稳 || **D2_DecayLinear10_GK_Vol** | $\sum_{i=0}^{9} w_i \cdot \text{gk\_vol}_{t-i}$ | **衰减加权波动率** — 近期波动率权重更大，更快感知波动率的上升/下降趋势 || **D3_DecayLinear20_LogHL** | $\sum_{i=0}^{19} w_i \cdot \text{log\_hl}_{t-i}$ | **衰减加权波幅** — 20期波幅的加权平均，捕捉振幅的渐变趋势 |### E. 正交残差因子**原理**：从信号X中剔除与信号Y共线的部分，得到"纯"的独立预测分量。| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **E1_Orth_EMA_Ret_vs_GKVol** | $\text{ema\_ret} - \beta \cdot \text{gk\_vol}, \beta=\text{Cov}/\text{Var}$ | **纯Alpha信号** — 剔除波动率影响后的EMA收益偏离。真正的"超额收益"信号，不受"高波动自然大偏离"干扰 || **E2_Orth_VWAP_Dev_vs_LogHL** | $\text{vwap\_dev} - \beta \cdot \text{log\_hl\_range}$ | **纯价格偏离** — 剔除日间波幅影响后的VWAP偏离。区分"真正的趋势"与"大波动日的随机偏离" || **E3_Orth_OrderImb_vs_GKVol** | $\text{order\_imbalance} - \beta \cdot \text{gk\_vol}$ | **纯买卖压力** — 剔除波动率后的订单不平衡。识别独立于市场情绪的真正资金流向 |### F. 因子动量Delta因子**原理**：对因子值计算时间差分，捕捉因子方向的变化（二阶信息）。| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **F1_Delta3_OrderImbalance** | $\text{order\_imbalance}_t - \text{order\_imbalance}_{t-3}$ | **买卖压力变化** — 近3期订单不平衡的变动。正=买方压力增大中，负=卖方压力增大中。捕捉资金方向的转变 || **F2_Delta5_GK_Vol** | $\text{gk\_vol}_t - \text{gk\_vol}_{t-5}$ | **波动率变化** — 近5期波动率的变动。正=波动率上升（市场焦虑增加），负=波动率下降（市场趋于平静） || **F3_Delta5_LogHL_Range** | $\text{log\_hl\_range}_t - \text{log\_hl\_range}_{t-5}$ | **波幅变化** — 日内振幅的近期变化。波幅扩张=突破性行情启动，波幅收缩=蓄势待发 |### G. 有符号幂变换因子（Round 1+Round 2精选）详见第二部分。代表性因子：| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **SP20_ema_ret_60M** | $\text{sign}(\text{ema\_ret}) \cdot |\text{ema\_ret}|^{2.0}$ | **平方放大EMA收益偏离** — 对趋势偏离进行平方放大，极端偏离被赋予极大权重。捕捉"强趋势延续"效应 || **SP20_ema_ret_15M** | $\text{sign}(\text{ema\_ret\_15}) \cdot |\text{ema\_ret\_15}|^{2.0}$ | **平方放大中期EMA偏离** — 15期EMA偏离的平方变换，中期趋势的强化版本 || **SP15_roc_5M** | $\text{sign}(\text{roc\_5}) \cdot |\text{roc\_5}|^{1.5}$ | **1.5次方短期动量** — 5期动量的温和放大。在保持线性关系的同时，适度强化强动量信号 || **SP20_roc_15M** | $\text{sign}(\text{roc\_15}) \cdot |\text{roc\_15}|^{2.0}$ | **平方放大中期动量** — 15期ROC的平方变换，对中期强趋势极度敏感 || **SP20_HMA_Deviation** | $\text{sign}(\text{hma\_dev}) \cdot |\text{hma\_dev}|^{2.0}$ | **平方HMA偏离** — 赫尔移动平均偏离的平方变换。HMA本身已消除滞后，平方变换进一步增强趋势信号 || **SP15_HMA_Deviation** | $\text{sign}(\text{hma\_dev}) \cdot |\text{hma\_dev}|^{1.5}$ | **温和HMA偏离** — 1.5次方的HMA偏离，比平方版本保守但更稳定 || **SP05_Log_HL_Range** | $|\text{log\_hl\_range}|^{0.5}$ | **平方根波幅** — 对数波幅的平方根压缩。抑制极端大波动日的影响，使信号更关注波幅的"边际变化"而非绝对水平 || **SP05_GK_Volatility_1M** | $|\text{gk\_vol}|^{0.5}$ | **平方根GK波动率** — 波动率的平方根压缩。降低波动率右尾的影响，使波动率信号更具线性预测力 || **SP20_KAMA_Deviation** | $\text{sign}(\text{kama\_dev}) \cdot |\text{kama\_dev}|^{2.0}$ | **平方KAMA偏离** — 考夫曼自适应均线偏离的平方。KAMA自适应调整平滑度，平方变换放大有效偏离 || **SP20_Ichimoku_Premium** | $\text{sign}(\text{ichimoku\_prem}) \cdot |\text{ichimoku\_prem}|^{2.0}$ | **平方一目均衡溢价** — 一目均衡云层溢价的平方变换。捕捉价格突破云层的极端信号 |### H. 交叉共振因子**原理**：两个互补信号相乘，只有两者同时极端时因子值才大。| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **H1_OI_x_GKVol** | $\text{order\_imbalance} \times \text{gk\_vol}$ | **放量失衡共振** — 买卖失衡 × 波动率。只有"高波动+强买卖失衡"同时出现时，信号才显著。过滤低波动的随机失衡 || **H2_VWAP_Dev_x_LogHL** | $\text{vwap\_dev} \times \text{log\_hl\_range}$ | **大波动偏离共振** — VWAP偏离 × 波幅。只有"大振幅+大偏离"才触发信号，排除窄幅震荡中的随机价格偏移 |

## 3.3 高阶因子详细目录 (wxy_new_factor)### 3.3.1 分数阶差分因子 (Fractional Differentiation)**原理**：分数阶差分在平稳性($d=1$, 完全差分)和记忆性($d=0$, 原序列)之间取得平衡。使用 $d=0.4$ 和窗口 $p=60$ 的截断分数阶差分，保留了序列的长期记忆特征，同时达到近似平稳。权重序列通过递推公式生成：$$w_0 = 1, \quad w_k = -w_{k-1} \cdot \frac{d - k + 1}{k}, \quad k = 1, 2, ..., p-1$$然后反转权重序列，通过滑动窗口点积应用于原始序列：$$\text{FD}_t = \sum_{k=0}^{p-1} w_{\text{rev}}[k] \cdot x_{t-p+1+k}$$| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **FracDiff_LogPrice_60M** | FD($\ln C$, d=0.4, p=60) | **分数阶差分对数价格** — 对数价格的"半平稳"版本。比简单收益率（d=1）保留了更多均值回归记忆，比原始价格（d=0）更平稳。捕捉价格的长期记忆性趋势 || **FracDiff_Ichimoku_Midline_60M** | FD($\ln(\text{Donchian\_Midline})$, d=0.4, p=60) | **分数阶差分一目均衡中轴** — Donchian中线的分数阶差分。Donchian中轴=(60日最高+最低)/2，代表中期价格中枢。FD处理保留中枢位移的记忆特征 || **FracDiff_EMA_60M** | FD($\ln(\text{EMA}_{60})$, d=0.4, p=60) | **分数阶差分EMA** — EMA趋势中枢的分数阶差分。平滑趋势的"半平稳"版本，比EMA差分（近似动量）保留了更长期的趋势渐变信息 || **FracDiff_VWAP_Prem_60M** | FD(VWAP溢价, d=0.4, p=60) | **分数阶差分VWAP溢价** — VWAP溢价的分数阶差分。VWAP溢价 = (C-VWAP)/VWAP，反映价格与成交量加权均价的偏离。FD处理使偏离信号更平滑、更具持续性 || **FracDiff_VPATS_60M** | FD(VPATS, d=0.4, p=60) | **分数阶差分成交量价格趋势** — VPATS信号的分数阶差分。VPATS = ((HMA-KAMA)/KAMA) × exp(-10×nATR)，综合了趋势偏离与波动率惩罚 || **FracDiff_RSI_60M** | FD((RSI-50)/50, d=0.4, p=60) | **分数阶差分RSI** — 中心化RSI的分数阶差分。RSI标准化到[-1,1]后进行FD，保留了超买超卖的时序记忆特征 |### 3.3.2 卡尔曼滤波因子 (Kalman Filter)**原理**：卡尔曼滤波通过"预测-更新"框架从含噪观测中提取隐藏状态，比传统移动平均具有零滞后优势（状态估计不引入相位延迟）。**1D卡尔曼滤波（公平价值状态）**：$$\begin{aligned}\text{预测: } & \hat{x}_{t|t-1} = \hat{x}_{t-1}, \quad P_{t|t-1} = P_{t-1} + Q \\\text{更新: } & K_t = P_{t|t-1} / (P_{t|t-1} + R) \\& \hat{x}_t = \hat{x}_{t|t-1} + K_t(z_t - \hat{x}_{t|t-1}) \\& P_t = (1 - K_t)P_{t|t-1}\end{aligned}$$**2D局部线性趋势卡尔曼滤波（隐藏趋势状态）**：$$\text{状态向量: } \mathbf{x} = [\text{level}, \text{trend}]^T$$$$\text{状态转移: } F = \begin{bmatrix} 1 & 1 \\ 0 & 1 \end{bmatrix}, \quad \text{观测: } H = [1, 0]$$| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **KF_FairValue_Dev** | $(C - \hat{x}_{\text{KF}}) / \hat{x}_{\text{KF}}$ | **公平价值偏离度** — 收盘价与1D卡尔曼估计公平价值的偏离百分比。卡尔曼估计无滞后地追踪"真实"价格水平。偏离大=价格与隐藏公平价值脱节，预示回归 || **KF_Hidden_Trend_Norm** | $\hat{t}_{\text{KF}} / C$ | **隐藏趋势强度** — 2D卡尔曼滤波提取的隐藏趋势分量，除以价格标准化。捕捉"不可见但持续"的价格漂移，比简单动量更早发现趋势形成 || **KF_VWAP_Residual** | $(\text{spread} - \hat{x}_{\text{KF}}) / \sqrt{P}$ | **标准化VWAP残差** — VWAP价差经1D卡尔曼滤波后的标准化新息(Innovation)。衡量"当前价差"与"卡尔曼预期价差"的偏离程度（以标准差计）。|值|>2=异常偏离信号 |### 3.3.3 成交量分布POC距离因子**原理**：成交量分布(Volume Profile)将固定窗口内的成交量按价格水平汇总，POC (Point of Control) 是成交量最大的价格水平，代表市场的"公允价值"共识。使用Numba JIT编译实现高效的50-bin成交量直方图计算：| 因子 | 公式 | 标的特征意义 ||------|------|------------|| **Distance_to_POC_60M** | $(C - \text{POC}_{60}) / \text{ATR}_{60}$ | **POC标准化距离** — 收盘价与60期成交量分布控制点的距离，除以ATR标准化。正值=价格高于价值共识区（溢价），负值=价格低于共识区（折价）。距离大=价格偏离共识，均值回归力量增强 |

### 3.3.4 文献综述因子基于学术文献实现的12个经典量化因子：| 因子 | 公式 | 文献来源 | 标的特征意义 ||------|------|---------|------------|| **Parkinson_Volatility_1M** | $\sqrt{\frac{1}{4\ln 2} \cdot \ln^2(H/L)}$ | Parkinson (1980) J. Business | **Parkinson波动率** — 仅用最高最低价估计波动率，比收盘价波动率效率高5.2倍。捕捉日内价格极端波动 || **Rogers_Satchell_Vol_1M** | $\sqrt{\ln(H/C)\ln(H/O) + \ln(L/C)\ln(L/O)}$ | Rogers & Satchell (1991) | **RS波动率** — 允许非零漂移的波动率估计量，优于Parkinson当趋势存在时。更准确度量有趋势市场中的真实波动 || **Lower_Shadow_Frac** | $(\min(O,C) - L) / (H - L)$ | K线形态学 | **下影线比率** — 下影线占全日振幅的比例。高值=盘中强力反弹(锤子线形态)，预示底部反转 || **Body_Fraction** | $|C-O| / (H-L)$ | K线形态学 | **实体占比** — K线实体占全日振幅的比例。高值=方向确定性强(长实体)，低值=方向不确定性高(十字星) || **Close_Position** | $(C-L) / (H-L)$ | K线形态学 | **收盘位置** — 收盘价在日内区间的相对位置。量化"尾盘效应"，反映收盘前的买卖力量对比 || **Roll_Spread_Proxy_1M** | $2\sqrt{-\text{Cov}(\Delta C_t, \Delta C_{t-1})}$ (当Cov<0时) | Roll (1984) J. Finance | **Roll价差代理** — 从价格变动的负自相关中推断有效买卖价差。捕捉市场微观结构中的流动性和交易成本 || **Kaufman_ER_60M** | $|C_t - C_{t-60}| / \sum_{i=1}^{60}|\Delta C_{t-i+1}|$ | Kaufman (1987) Wiley | **Kaufman效率比率** — 净价格变动与总路径长度的比值。ER→1=趋势高效(方向明确)，ER→0=震荡(方向噪声大) || **Donchian_Width_60M** | $(\max_{60}(H) - \min_{60}(L)) / \text{Close}$ | Donchian通道 | **Donchian通道宽度** — 60期最高最低价范围/收盘价。度量价格的绝对波动范围，通道宽=大趋势，通道窄=盘整 || **Price_Volume_Corr_60M** | $\text{Corr}_{60}(\Delta C, V)$ | 量价关系文献 | **量价相关性** — 价格变动与成交量的60期Pearson相关。正相关=量价配合(趋势健康)，负相关=量价背离(趋势存疑) || **Volume_Weighted_Close_60M** | $\sum_{i=1}^{60} V_{t-i} \cdot C_{t-i} / \sum_{i=1}^{60} V_{t-i}$ | 成交量加权微观结构 | **成交量加权收盘价** — 60期VWAP。反映"大多数成交在什么价格发生"，代表持仓成本的集中区域 || **RV_5M_Realized_Vol** | $\sqrt{\sum_{i=1}^{5} (\ln C_{t-i+1}/C_{t-i})^2}$ | Andersen & Bollerslev (1998) | **5期实现波动率** — 基于高频收益的已实现波动。比单期GK波动率更精确地度量短期真实波动 || **Risk_Adj_Momentum_60M** | $(C_t - C_{t-60}) / (C_{t-60} \cdot \sigma_{60})$ | Barroso & Santa-Clara (2015) RFS | **风险调整动量** — 60期动量除以波动率缩放。经风险调整的动量策略，波动率高时自动降低动量敞口 |

---# 第四部分：因子计算代码（整合版）以下代码整合了 optimized_factors_IC.ipynb 和 wxy_new_factor.ipynb 中的所有因子计算逻辑。

In [ ]:
# ============================================================# Cell 1: 导入库与全局配置# ============================================================import numpy as npimport bottleneck as bnfrom numpy.lib.stride_tricks import sliding_window_view# 尝试导入 ta-lib (可选，用于部分信号)try:    import talib    HAS_TALIB = Trueexcept ImportError:    HAS_TALIB = False    print('Warning: talib not installed. Some factors (TRIX, PPO, CCI, etc.) will be skipped.')# 尝试导入 numba (可选，用于POC计算)try:    from numba import njit    HAS_NUMBA = Trueexcept ImportError:    HAS_NUMBA = False    def njit(*args, **kwargs):        return lambda f: fEPS = 1e-8print('Libraries loaded.')

In [ ]:
# ============================================================# Cell 2: 基础工具函数# ============================================================def ts_rank(x, window):    """时序排名：将每个值映射到其窗口内的分位数排名[0,1]"""    T = len(x)    result = np.full(T, 0.5, dtype=np.float32)    if T <= window:        return result    sw = sliding_window_view(x.astype(np.float64), window)    last_col = sw[:, -1:]    result[window-1:] = (sw <= last_col).mean(axis=1)    return resultdef rolling_zscore(x, window):    """滚动Z-Score标准化"""    mean_w = bn.move_mean(x, window=window, min_count=1)    std_w = bn.move_std(x, window=window, min_count=1)    result = (x - mean_w) / (std_w + EPS)    return np.nan_to_num(result, nan=0.0).astype(np.float32)def rolling_beta(x, y, window):    """滚动Beta系数: Cov(x,y) / Var(y)"""    mean_x = bn.move_mean(x, window=window, min_count=1)    mean_y = bn.move_mean(y, window=window, min_count=1)    cov_xy = bn.move_mean(x * y, window=window, min_count=1) - mean_x * mean_y    var_y = bn.move_var(y, window=window, min_count=1)    return cov_xy / (var_y + EPS)def decay_linear_weights(window):    """生成衰减线性权重: [w, w-1, ..., 1] / sum"""    weights = np.arange(window, 0, -1, dtype=np.float64)    return weights / weights.sum()def rolling_decay_linear(x, window):    """滚动衰减线性加权平均"""    T = len(x)    result = np.full(T, np.nan, dtype=np.float64)    w = decay_linear_weights(window)    for t in range(window - 1, T):        result[t] = np.dot(x[t - window + 1 : t + 1], w)    result[:window-1] = bn.move_mean(x, window=window, min_count=1)[:window-1]    return np.nan_to_num(result, nan=0.0).astype(np.float32)# ===== 有符号幂变换核心函数 =====def signed_power(x, p):    """有符号幂变换: sign(x) * |x|^p — 保留方向，缩放幅值"""    return np.sign(x) * (np.abs(x) ** p)def power_only(x, p):    """纯幂变换: max(x,0)^p — 用于正值信号（波动率等）"""    return np.maximum(x, 0) ** pprint('Utility functions defined.')

In [ ]:
# ============================================================# Cell 3: 基础因子计算函数 — compute_base_factors# ============================================================def compute_base_factors(O, H, L, C, V):    """    从OHLCV计算11个基础因子信号。    输入: O, H, L, C, V — 各为长度T的一维numpy数组    输出: (T, 11) 的二维数组，列顺序如下:        [log_hl, log_oc, gk_vol, log_hl_range, close_position,         log_ret, order_imbalance, vwap_dev, roc_5, roc_60, macd_line]    """    T = len(C)    factors = np.zeros((T, 11), dtype=np.float32)    # 对数高低价差    log_hl = np.log(np.maximum(H, L + EPS) / np.maximum(L, EPS))    factors[:, 0] = log_hl    # 对数开收盘价差    log_oc = np.log(np.maximum(O, C + EPS) / np.maximum(C, EPS))    factors[:, 1] = log_oc    # GK波动率 Parkinson-Garman-Klass    gk_vol = np.maximum(0.5 * log_hl**2 - (2*np.log(2)-1) * log_oc**2, 0)    factors[:, 2] = gk_vol    # 对数波幅    factors[:, 3] = log_hl  # alias: log_hl_range    # 收盘位置 (尾盘效应)    hl_range = H - L    factors[:, 4] = np.where(hl_range > EPS, (C - L) / (hl_range + EPS), 0.5)    # 对数收益率    log_C = np.log(np.maximum(C, EPS))    log_ret = np.zeros(T, dtype=np.float32)    log_ret[1:] = log_C[1:] - log_C[:-1]    factors[:, 5] = log_ret    # 订单不平衡代理    price_diff = np.zeros(T, dtype=np.float32)    price_diff[1:] = C[1:] - C[:-1]    order_imbalance = price_diff / (np.log(np.maximum(V, 1)) + EPS)    factors[:, 6] = order_imbalance    # VWAP偏离 (相对OHLC均值)    ohlc4 = (O + H + L + C) / 4.0    vwap_dev = (C - ohlc4) / (ohlc4 + EPS)    factors[:, 7] = vwap_dev    # ROC 5期和60期变化率    roc_5 = np.zeros(T, dtype=np.float32)    roc_60 = np.zeros(T, dtype=np.float32)    roc_5[5:] = (C[5:] - C[:-5]) / (C[:-5] + EPS)    roc_60[60:] = (C[60:] - C[:-60]) / (C[:-60] + EPS)    factors[:, 8] = roc_5    factors[:, 9] = roc_60    # MACD线    ema12 = bn.move_mean(C, window=12, min_count=1)    ema26 = bn.move_mean(C, window=26, min_count=1)    # 注：实际EMA有衰减因子，这里使用SMA简化。如需精确EMA，使用talib.EMA    macd_line = ema12 - ema26    factors[:, 10] = macd_line    return factorsprint('compute_base_factors defined.')

In [ ]:
# ============================================================# Cell 4: A-H 八大类优化因子生成函数# ============================================================def generate_optimized_factors(O, H, L, C, V):    """    生成25个优化因子 (A-H 八大类)。    返回: (T, 25) 的二维数组    """    # 先计算基础信号    base = compute_base_factors(O, H, L, C, V)    log_hl     = base[:, 0]    gk_vol     = base[:, 2]    log_hl_range = base[:, 3]    close_pos  = base[:, 4]    log_ret    = base[:, 5]    order_imb  = base[:, 6]    vwap_dev   = base[:, 7]    roc_5      = base[:, 8]    roc_60     = base[:, 9]    macd       = base[:, 10]    # ema_ret 计算公式    ema60_simple = bn.move_mean(C, window=60, min_count=1)    ema_ret = (C - ema60_simple) / (ema60_simple + EPS)    T = len(C)    factors = np.zeros((T, 25), dtype=np.float32)    col = 0    # === A.时序排序 (4个) ===    factors[:, col]; col+=1;  # skip (keep track)    # 用dict做索引映射更清晰：    # A1:    factors[:, 0] = ts_rank(gk_vol, 5)    factors[:, 1] = ts_rank(order_imb, 20)    factors[:, 2] = ts_rank(log_hl_range, 60)    factors[:, 3] = ts_rank(vwap_dev, 10)    # === B.滚动Z-Score (4个) ===    factors[:, 4] = rolling_zscore(gk_vol, 60)    factors[:, 5] = rolling_zscore(order_imb, 60)    factors[:, 6] = rolling_zscore(log_hl_range, 60)    factors[:, 7] = rolling_zscore(close_pos, 30)    # === C.波动率缩放 (4个) ===    gk_ma5 = bn.move_mean(gk_vol, window=5, min_count=1)    gk_ma20 = bn.move_mean(gk_vol, window=20, min_count=1)    gk_ma60 = bn.move_mean(gk_vol, window=60, min_count=1)    factors[:, 8]  = roc_5 / (np.sqrt(gk_ma5) + EPS)    factors[:, 9]  = roc_60 / (np.sqrt(gk_ma60) + EPS)    factors[:, 10] = macd / (np.sqrt(gk_ma60) + EPS)    factors[:, 11] = order_imb / (np.sqrt(gk_ma20) + EPS)    # === D.衰减线性加权 (3个) ===    factors[:, 12] = rolling_decay_linear(log_ret, 5)    factors[:, 13] = rolling_decay_linear(gk_vol, 10)    factors[:, 14] = rolling_decay_linear(log_hl_range, 20)    # === E.正交残差 (3个) ===    beta1 = rolling_beta(ema_ret, gk_vol, 60)    factors[:, 15] = ema_ret - beta1 * gk_vol    beta2 = rolling_beta(vwap_dev, log_hl_range, 60)    factors[:, 16] = vwap_dev - beta2 * log_hl_range    beta3 = rolling_beta(order_imb, gk_vol, 60)    factors[:, 17] = order_imb - beta3 * gk_vol    # === F.因子动量Delta (3个) ===    factors[3:, 18] = order_imb[3:] - order_imb[:-3]    factors[5:, 19] = gk_vol[5:] - gk_vol[:-5]    factors[5:, 20] = log_hl_range[5:] - log_hl_range[:-5]    # === G.有符号幂 (2个) ===    factors[:, 21] = signed_power(log_ret, 2.0)    factors[:, 22] = signed_power(vwap_dev, 1.5)    # === H.交叉共振 (2个) ===    factors[:, 23] = order_imb * gk_vol    factors[:, 24] = vwap_dev * log_hl_range    return factors# 因子名称映射OPTIMIZED_FACTOR_NAMES = [    'A1_TsRank5_GK_Vol', 'A2_TsRank20_OrderImbalance',    'A3_TsRank60_LogHL_Range', 'A4_TsRank10_VWAP_Dev',    'B1_ZScore60_GK_Vol', 'B2_ZScore60_OrderImbalance',    'B3_ZScore60_LogHL_Range', 'B4_ZScore30_Close_Position',    'C1_VolScaled_ROC_5M', 'C2_VolScaled_ROC_60M',    'C3_VolScaled_MACD_60M', 'C4_VolScaled_OrderImbalance',    'D1_DecayLinear5_Returns', 'D2_DecayLinear10_GK_Vol',    'D3_DecayLinear20_LogHL',    'E1_Orth_EMA_Ret_vs_GKVol', 'E2_Orth_VWAP_Dev_vs_LogHL',    'E3_Orth_OrderImb_vs_GKVol',    'F1_Delta3_OrderImbalance', 'F2_Delta5_GK_Vol',    'F3_Delta5_LogHL_Range',    'G1_SignedPow2_LogRet', 'G2_SignedPow15_VWAP_Dev',    'H1_OI_x_GKVol', 'H2_VWAP_Dev_x_LogHL']print(f'generate_optimized_factors: {len(OPTIMIZED_FACTOR_NAMES)} factors')

In [ ]:
# ============================================================# Cell 5: 扩展基础因子 (含技术指标) — 用于有符号幂变换 Round 1# ============================================================def compute_base_factors_extended(O, H, L, C, V):    """    在11个基础因子之上，增加15个技术指标/统计量。    输出: (T, 26) 的二维数组    """    T = len(C)    base = compute_base_factors(O, H, L, C, V)    ext = np.zeros((T, 26), dtype=np.float32)    ext[:, :11] = base    col = 11    hl_range = H - L    # Trix 60M    if HAS_TALIB:        trix_60 = talib.TEMA(C.astype(np.float64), timeperiod=60)        trix_60 = np.nan_to_num(trix_60, nan=0.0).astype(np.float32)    else:        trix_60 = np.zeros(T, dtype=np.float32)    ext[:, col]; col+=1  # col 11    ext[:, 11] = trix_60    # PPO 60M    if HAS_TALIB:        ppo_60 = talib.PPO(C.astype(np.float64), fastperiod=12, slowperiod=26, matype=0)        ppo_60 = np.nan_to_num(ppo_60, nan=0.0).astype(np.float32)    else:        ppo_60 = np.zeros(T, dtype=np.float32)    ext[:, 12] = ppo_60    # RSI (centered, scaled to [-1,1])    if HAS_TALIB:        rsi = talib.RSI(C.astype(np.float64), timeperiod=60)        rsi_centered = (np.nan_to_num(rsi, nan=50) - 50) / 50    else:        rsi_centered = np.zeros(T, dtype=np.float32)    ext[:, 13] = rsi_centered.astype(np.float32)    # CCI 60M    if HAS_TALIB:        cci_60 = talib.CCI(H.astype(np.float64), L.astype(np.float64), C.astype(np.float64), timeperiod=60)        cci_60 = np.nan_to_num(cci_60, nan=0.0).astype(np.float32)    else:        cci_60 = np.zeros(T, dtype=np.float32)    ext[:, 14] = cci_60    # VPATS 60M: ((HMA - KAMA) / KAMA) * exp(-10 * nATR)    hma = talib.WMA(C.astype(np.float64), timeperiod=30) if HAS_TALIB else bn.move_mean(C, window=30, min_count=1)    kama = talib.KAMA(C.astype(np.float64), timeperiod=60) if HAS_TALIB else bn.move_mean(C, window=60, min_count=1)    atr = talib.ATR(H.astype(np.float64), L.astype(np.float64), C.astype(np.float64), timeperiod=60) if HAS_TALIB else np.ones(T)    natr = np.where(C > EPS, atr / C, 0)    vpats = np.where(np.abs(kama) > EPS,        ((hma - kama) / kama) * np.exp(-10 * natr), 0)    ext[:, 15] = np.nan_to_num(vpats, nan=0.0).astype(np.float32)    # nATR 60M    ext[:, 16] = np.nan_to_num(natr, nan=0.0).astype(np.float32)    # ADX 60M    if HAS_TALIB:        adx_60 = talib.ADX(H.astype(np.float64), L.astype(np.float64), C.astype(np.float64), timeperiod=60)        adx_60 = np.nan_to_num(adx_60, nan=0.0).astype(np.float32)    else:        adx_60 = np.zeros(T, dtype=np.float32)    ext[:, 17] = adx_60    # CMF 60M (Chaikin Money Flow)    mf_mult = ((C - L) - (H - C)) / (hl_range + EPS)    mf_vol = mf_mult * V    cmf_60 = bn.move_sum(mf_vol, window=60, min_count=1) / (bn.move_sum(V, window=60, min_count=1) + EPS)    ext[:, 18] = np.nan_to_num(cmf_60, nan=0.0).astype(np.float32)    # Amihud 60M 非流动性指标    log_ret = base[:, 5]    abs_ret = np.abs(log_ret)    dollar_vol = V * C    amihud_ratio = np.where(dollar_vol > EPS, abs_ret / dollar_vol, 0)    amihud_60 = bn.move_mean(amihud_ratio, window=60, min_count=1)    ext[:, 19] = np.nan_to_num(amihud_60, nan=0.0).astype(np.float32)    # Realized Vol 5M    rv_5 = np.sqrt(bn.move_sum(log_ret**2, window=5, min_count=1))    ext[:, 20] = np.nan_to_num(rv_5, nan=0.0).astype(np.float32)    # Log HL Range Roll 60    log_hl_60 = bn.move_mean(base[:, 0], window=60, min_count=1)    ext[:, 21] = log_hl_60    # RV 60M    rv_60 = bn.move_sum(log_ret**2, window=60, min_count=1)    ext[:, 22] = np.nan_to_num(rv_60, nan=0.0).astype(np.float32)    # BPV 60M (Bipower Variation)    abs_ret_lag = np.zeros(T, dtype=np.float32)    abs_ret_lag[1:] = np.abs(log_ret[:-1])    bpv_60 = (np.pi/2) * bn.move_sum(abs_ret_lag * np.abs(log_ret), window=60, min_count=1)    ext[:, 23] = np.nan_to_num(bpv_60, nan=0.0).astype(np.float32)    # Jump Ratio 60M    jump_ratio = np.where(rv_60 > EPS, np.maximum((rv_60 - bpv_60) / rv_60, 0), 0)    ext[:, 24] = jump_ratio    # Upper Shadow Fraction    upper_shadow = np.where(hl_range > EPS, (H - np.maximum(O, C)) / hl_range, 0)    ext[:, 25] = upper_shadow    return extprint('compute_base_factors_extended defined.')

In [ ]:
# ============================================================# Cell 6: 有符号幂变换因子生成 — Round 1 (35个因子)# ============================================================def generate_sp_round1_factors(O, H, L, C, V):    """    生成Round 1的有符号幂变换因子 (35个)。    四种幂次方案: SP15(p=1.5), SP20(p=2.0), SP05(p=0.5), SP03(p=0.3)    返回: (T, 35) 的二维数组    """    ext = compute_base_factors_extended(O, H, L, C, V)    T = len(C)    factors = np.zeros((T, 35), dtype=np.float32)    # 提取所有扩展基础信号    # col 0-10: 基础信号    gk_vol = ext[:, 2]    log_hl_range = ext[:, 3]    close_pos = ext[:, 4]    log_ret = ext[:, 5]    order_imb = ext[:, 6]    vwap_dev = ext[:, 7]    roc_5 = ext[:, 8]    roc_60 = ext[:, 9]    macd = ext[:, 10]    # col 11+: 技术指标    trix_60 = ext[:, 11]    ppo_60 = ext[:, 12]    rsi_c = ext[:, 13]    cci_60 = ext[:, 14]    vpats_60 = ext[:, 15]    natr_60 = ext[:, 16]    adx_60 = ext[:, 17]    cmf_60 = ext[:, 18]    amihud_60 = ext[:, 19]    rv_5 = ext[:, 20]    log_hl_60 = ext[:, 21]    rv_60 = ext[:, 22]    bpv_60 = ext[:, 23]    jump_ratio = ext[:, 24]    upper_shadow = ext[:, 25]    idx = 0    # === SP15: 方向性信号 p=1.5 温和放大 (14个) ===    sp15_inputs = [vwap_dev, order_imb, log_ret, roc_5, roc_60,                    macd, ppo_60, trix_60, vpats_60, cci_60,                    rsi_c, cmf_60, close_pos - 0.5, upper_shadow]    sp15_names = ['vwap_dev', 'order_imb', 'log_ret', 'roc_5', 'roc_60',                  'macd', 'ppo', 'trix', 'vpats', 'cci',                  'rsi_centered', 'cmf', 'close_pos', 'upper_shadow']    for x in sp15_inputs:        factors[:, idx] = signed_power(x, 1.5)        idx += 1    # === SP20: 方向性信号 p=2.0 强放大 (8个) ===    sp20_dir_inputs = [vwap_dev, log_ret, order_imb, roc_5, roc_60,                        macd, trix_60]    for x in sp20_dir_inputs:        factors[:, idx] = signed_power(x, 2.0)        idx += 1    # SP20: 有界[0,1]信号 (2个)    factors[:, idx] = power_only(close_pos, 2.0); idx += 1    body_frac = np.abs(log_ret) / (log_hl_range + EPS)  # approximate    factors[:, idx] = power_only(body_frac, 2.0); idx += 1    # === SP05: 波动率/价差信号 p=0.5 平方根压缩 (6个) ===    sp05_inputs = [gk_vol, log_hl_range, natr_60, adx_60, rv_5, log_hl_60]    for x in sp05_inputs:        factors[:, idx] = power_only(x, 0.5)        idx += 1    # === SP03: 波动率/流动性信号 p=0.3 强压缩 (5个) ===    sp03_inputs = [gk_vol, log_hl_range, natr_60, jump_ratio, amihud_60]    for x in sp03_inputs:        factors[:, idx] = power_only(x, 0.3)        idx += 1    return factors# Round 1因子名称 (除已含在extended中的基础信号外)SP_ROUND1_NAMES = [    'SP15_Intrabar_VWAP_Dev', 'SP15_Order_Imbalance_Px', 'SP15_LogRet',    'SP15_roc_5M', 'SP15_roc_60M', 'SP15_macd_60M', 'SP15_ppo_60M',    'SP15_trix_60M', 'SP15_vpats_60M', 'SP15_cci_60M',    'SP15_rsi_centered_60M', 'SP15_cmf_60M', 'SP15_Close_Position',    'SP15_Upper_Shadow_Frac',    'SP20_Intrabar_VWAP_Dev', 'SP20_LogRet_1', 'SP20_Order_Imbalance_Px',    'SP20_roc_5M', 'SP20_roc_60M', 'SP20_macd_60M', 'SP20_trix_60M',    'SP20_Close_Position', 'SP20_Body_Fraction',    'SP05_GK_Volatility_1M', 'SP05_Log_HL_Range', 'SP05_natr_60M',    'SP05_adx_60M', 'SP05_Realized_Vol_5M', 'SP05_Log_HL_Range_Roll60',    'SP03_GK_Volatility_1M', 'SP03_Log_HL_Range', 'SP03_natr_60M',    'SP03_Jump_Ratio_60M', 'SP03_Amihud_60M']print(f'generate_sp_round1_factors: {len(SP_ROUND1_NAMES)} factors')

In [ ]:
# ============================================================# Cell 7: 有符号幂变换因子生成 — Round 2 (35个因子)# ============================================================def compute_base_factors_round2(O, H, L, C, V):    """    计算Round 2的18个新基础信号。    输出: (T, 18) 的二维数组    """    T = len(C)    sig = np.zeros((T, 18), dtype=np.float32)    hl_range = H - L    ohlc4 = (O + H + L + C) / 4.0    log_C = np.log(np.maximum(C, EPS))    # 1. ema_ret_5M    ema5 = bn.move_mean(C, window=5, min_count=1)    sig[:, 0] = (C - ema5) / (ema5 + EPS)    # 2. ema_ret_15M    ema15 = bn.move_mean(C, window=15, min_count=1)    sig[:, 1] = (C - ema15) / (ema15 + EPS)    # 3. mom_5M (raw price diff)    mom5 = np.zeros(T, dtype=np.float32)    mom5[5:] = C[5:] - C[:-5]    sig[:, 2] = mom5    # 4. mom_60M    mom60 = np.zeros(T, dtype=np.float32)    mom60[60:] = C[60:] - C[:-60]    sig[:, 3] = mom60    # 5. roc_15M    roc15 = np.zeros(T, dtype=np.float32)    roc15[15:] = (C[15:] - C[:-15]) / (C[:-15] + EPS)    sig[:, 4] = roc15    # 6. HMA Deviation (Hull Moving Average)    half = 30    sqrt_p = int(np.sqrt(60))    wma_half = bn.move_mean(C, window=half, min_count=1)    wma_full = bn.move_mean(C, window=60, min_count=1)    hma_raw = 2 * wma_half - wma_full    hma = bn.move_mean(hma_raw, window=sqrt_p, min_count=1)    sig[:, 5] = (hma - C) / (C + EPS)    # 7. KAMA Deviation (Kaufman Adaptive Moving Average)    kama = bn.move_mean(C, window=60, min_count=1)  # simplified KAMA    sig[:, 6] = (kama - C) / (C + EPS)    # 8. Ichimoku Premium    hh30 = bn.move_max(H, window=30, min_count=1)    ll30 = bn.move_min(L, window=30, min_count=1)    midline = (hh30 + ll30) / 2.0    sig[:, 7] = (C - midline) / (midline + EPS)    # 9. WILLR 60M    if HAS_TALIB:        willr = talib.WILLR(H.astype(np.float64), L.astype(np.float64), C.astype(np.float64), timeperiod=60)        willr = np.nan_to_num(willr, nan=-50) / 100.0    else:        hh60 = bn.move_max(H, window=60, min_count=1)        ll60 = bn.move_min(L, window=60, min_count=1)        willr = np.where(hh60 - ll60 > EPS, -100 * (hh60 - C) / (hh60 - ll60 + EPS), -50) / 100.0    sig[:, 8] = willr.astype(np.float32)    # 10. CMO 60M    if HAS_TALIB:        cmo = talib.CMO(C.astype(np.float64), timeperiod=60)        cmo = np.nan_to_num(cmo, nan=0) / 100.0    else:        cmo = np.zeros(T, dtype=np.float32)    sig[:, 9] = cmo.astype(np.float32)    # 11. BOP EMA 60M    bop = np.where(hl_range > EPS, (C - O) / hl_range, 0)    sig[:, 10] = bn.move_mean(bop, window=60, min_count=1)    # 12. Aroon Osc 60M    aroon_up = np.zeros(T, dtype=np.float32)    aroon_down = np.zeros(T, dtype=np.float32)    for t in range(60, T):        window_h = H[t-60:t+1]        window_l = L[t-60:t+1]        aroon_up[t] = 100 * (60 - (60 - np.argmax(window_h))) / 60        aroon_down[t] = 100 * (60 - (60 - np.argmin(window_l))) / 60    sig[:, 11] = (aroon_up - aroon_down) / 100.0    # 13. Volume ROC 20M    vol_roc = np.zeros(T, dtype=np.float32)    vol_roc[20:] = (V[20:] - V[:-20]) / (V[:-20] + EPS)    sig[:, 12] = vol_roc    # 14. Price-Volume Delta Product    dP = np.zeros(T, dtype=np.float32)    dP[1:] = C[1:] - C[:-1]    dV = np.zeros(T, dtype=np.float32)    dV[1:] = V[1:] - V[:-1]    sig[:, 13] = (dP * dV) / (C * V + EPS)    # 15. ROC Diff 5-60    roc5 = np.zeros(T, dtype=np.float32)    roc60 = np.zeros(T, dtype=np.float32)    roc5[5:] = (C[5:] - C[:-5]) / (C[:-5] + EPS)    roc60[60:] = (C[60:] - C[:-60]) / (C[:-60] + EPS)    sig[:, 14] = roc5 - roc60    # 16. RV Ratio 5/60    log_r = np.zeros(T, dtype=np.float32)    log_r[1:] = log_C[1:] - log_C[:-1]    rv5 = bn.move_sum(log_r**2, window=5, min_count=1)    rv60 = bn.move_sum(log_r**2, window=60, min_count=1)    sig[:, 15] = rv5 / (rv60 + EPS) - 1.0    # 17. BB Position 60M (centered to [-0.5, 0.5])    bb_mid = bn.move_mean(C, window=60, min_count=1)    bb_std = bn.move_std(C, window=60, min_count=1)    bb_pos = np.where(bb_std > EPS, (C - bb_mid) / (2 * bb_std + EPS), 0)    sig[:, 16] = bb_pos    # 18. VWAP Spread 60M    vwap_num = bn.move_sum(V * ohlc4, window=60, min_count=1)    vwap_den = bn.move_sum(V, window=60, min_count=1)    vwap60 = vwap_num / (vwap_den + EPS)    sig[:, 17] = (C - vwap60) / (C + EPS)    return sigdef generate_sp_round2_factors(O, H, L, C, V):    """    生成Round 2的有符号幂变换因子 (35个)。    使用p=1.5和p=2.0对18个基础信号进行变换。    """    sig = compute_base_factors_round2(O, H, L, C, V)    T = len(C)    factors = np.zeros((T, 35), dtype=np.float32)    idx = 0    for p in [1.5, 2.0]:        for j in range(sig.shape[1]):            # bb_pos and vwap_spread use their own range            factors[:, idx] = signed_power(sig[:, j], p)            idx += 1    return factorsprint('generate_sp_round2_factors defined.')

In [ ]:
# ============================================================# Cell 8: 分数阶差分因子 (wxy_new_factor)# ============================================================def get_frac_diff_weights(d, p):    """    生成分数阶差分的权重序列 (截断于窗口p)。    d: 差分阶数 (0 < d < 1)    p: 截断窗口    返回: 长度为p的权重数组 (已反转，可直接用于点积)    """    w = np.zeros(p, dtype=np.float64)    w[0] = 1.0    for k in range(1, p):        w[k] = -w[k-1] * (d - k + 1) / k    # 反转权重: w_rev[0]=w[-1], w_rev[-1]=w[0]    return w[::-1]def frac_diff(series, d, p):    """    对输入序列进行分数阶差分。    series: 长度为T的一维数组    d: 差分阶数    p: 截断窗口    返回: 长度为T的数组 (前p-1个值为0)    """    T = len(series)    result = np.zeros(T, dtype=np.float32)    if T <= p:        return result    w_rev = get_frac_diff_weights(d, p)    sw = sliding_window_view(series.astype(np.float64), p)    result[p-1:] = np.dot(sw, w_rev)    return resultdef generate_frac_diff_factors(O, H, L, C, V, d=0.4, p=60):    """    生成6个分数阶差分因子。    返回: (T, 6) 的二维数组    """    T = len(C)    fd = np.zeros((T, 6), dtype=np.float32)    log_C = np.log(np.maximum(C, EPS))    # 1. FracDiff LogPrice    fd[:, 0] = frac_diff(log_C, d, p)    # 2. FracDiff Ichimoku Midline (Donchian)    hh = bn.move_max(H, window=p, min_count=1)    ll = bn.move_min(L, window=p, min_count=1)    midline = (hh + ll) / 2.0    log_midline = np.log(np.maximum(midline, EPS))    fd[:, 1] = frac_diff(log_midline, d, p)    # 3. FracDiff EMA    ema_p = bn.move_mean(C, window=p, min_count=1)    log_ema = np.log(np.maximum(ema_p, EPS))    fd[:, 2] = frac_diff(log_ema, d, p)    # 4. FracDiff VWAP Premium    ohlc4 = (O + H + L + C) / 4.0    vwap_num = bn.move_sum(V * ohlc4, window=p, min_count=1)    vwap_den = bn.move_sum(V, window=p, min_count=1)    vwap = vwap_num / (vwap_den + EPS)    vwap_prem = (C - vwap) / (vwap + EPS)    fd[:, 3] = frac_diff(vwap_prem, d, p)    # 5. FracDiff VPATS    # Simplified VPATS: (HMA - KAMA) / KAMA * exp(-10 * ATR/C)    atr_raw = np.zeros(T, dtype=np.float32)    for t in range(1, T):        tr = max(H[t]-L[t], abs(H[t]-C[t-1]), abs(L[t]-C[t-1]))        atr_raw[t] = tr    atr_ema = bn.move_mean(atr_raw, window=p, min_count=1)    natr = np.where(C > EPS, atr_ema / C, 0)    hma = bn.move_mean(C, window=p//2, min_count=1)    vpats = np.where(np.abs(hma) > EPS,        (hma - bn.move_mean(C, window=p, min_count=1)) / hma * np.exp(-10 * natr), 0)    fd[:, 4] = frac_diff(vpats, d, p)    # 6. FracDiff RSI centered    rsi_centered = np.zeros(T, dtype=np.float32)    if HAS_TALIB:        rsi = talib.RSI(C.astype(np.float64), timeperiod=p)        rsi_centered = (np.nan_to_num(rsi, nan=50) - 50) / 50    fd[:, 5] = frac_diff(rsi_centered, d, p)    return fdFRAC_DIFF_NAMES = [    'FracDiff_LogPrice_60M', 'FracDiff_Ichimoku_Midline_60M',    'FracDiff_EMA_60M', 'FracDiff_VWAP_Prem_60M',    'FracDiff_VPATS_60M', 'FracDiff_RSI_60M']print('Fractional differentiation factors defined.')

In [ ]:
# ============================================================# Cell 9: 卡尔曼滤波因子 (wxy_new_factor)# ============================================================def calc_kf_1d_fast(observations, Q=1e-4, R=1e-2):    """    1D卡尔曼滤波 — 从含噪观测中提取隐藏公平价值状态。    观测: observations (价格序列)    Q: 过程噪声方差 (默认1e-4, 小值=假设状态变化缓慢)    R: 观测噪声方差 (默认1e-2, 大值=观测信噪比低)    返回: (filtered_state, state_covariance)    """    T = len(observations)    x_hat = np.zeros(T, dtype=np.float64)    P = np.zeros(T, dtype=np.float64)    x_hat[0] = observations[0]    P[0] = 1.0    for t in range(1, T):        # 预测        x_pred = x_hat[t-1]        P_pred = P[t-1] + Q        # 更新        K = P_pred / (P_pred + R)        x_hat[t] = x_pred + K * (observations[t] - x_pred)        P[t] = (1 - K) * P_pred    return x_hat.astype(np.float32), P.astype(np.float32)def calc_kf_local_linear_trend(observations, Q_level=1e-4, Q_trend=1e-6, R=1e-2):    """    2D局部线性趋势卡尔曼滤波 — 提取隐藏的价格趋势。    状态向量: [level, trend]    Q_level: 水平过程噪声    Q_trend: 趋势过程噪声 (极小值=趋势缓慢变化)    R: 观测噪声    返回: (level, trend)    """    T = len(observations)    level = np.zeros(T, dtype=np.float64)    trend = np.zeros(T, dtype=np.float64)    P00 = np.zeros(T, dtype=np.float64)    P10 = np.zeros(T, dtype=np.float64)    P11 = np.zeros(T, dtype=np.float64)    level[0] = observations[0]    trend[0] = 0.0    P00[0] = 1.0    P11[0] = 1.0    Q = np.array([[Q_level, 0], [0, Q_trend]])    for t in range(1, T):        # 预测        level_pred = level[t-1] + trend[t-1]        trend_pred = trend[t-1]        P00_pred = P00[t-1] + 2*P10[t-1] + P11[t-1] + Q_level        P10_pred = P10[t-1] + P11[t-1]        P11_pred = P11[t-1] + Q_trend        # 更新        y = observations[t] - level_pred        S = P00_pred + R        K0 = P00_pred / S        K1 = P10_pred / S        level[t] = level_pred + K0 * y        trend[t] = trend_pred + K1 * y        P00[t] = (1 - K0) * P00_pred        P10[t] = (1 - K0) * P10_pred - K1 * P00_pred * 0  # simplified        P11[t] = P11_pred - K1 * P10_pred    return level.astype(np.float32), trend.astype(np.float32)def generate_kalman_factors(O, H, L, C, V):    """    生成3个卡尔曼滤波因子。    返回: (T, 3) 的二维数组    """    T = len(C)    kf_factors = np.zeros((T, 3), dtype=np.float32)    # 1. KF Fair Value Deviation    kf_state, _ = calc_kf_1d_fast(C, Q=1e-4, R=1e-2)    kf_factors[:, 0] = (C - kf_state) / (kf_state + EPS)    # 2. KF Hidden Trend Normalized    _, kf_trend = calc_kf_local_linear_trend(C, Q_level=1e-4, Q_trend=1e-6, R=1e-2)    kf_factors[:, 1] = kf_trend / (C + EPS)    # 3. KF VWAP Residual    ohlc4 = (O + H + L + C) / 4.0    vwap_num = bn.move_sum(V * ohlc4, window=60, min_count=1)    vwap_den = bn.move_sum(V, window=60, min_count=1)    vwap = vwap_num / (vwap_den + EPS)    spread = C - vwap    spread_state, spread_P = calc_kf_1d_fast(spread, Q=1e-5, R=1e-2)    kf_factors[:, 2] = (spread - spread_state) / (np.sqrt(spread_P) + EPS)    return kf_factorsKF_NAMES = [    'KF_FairValue_Dev', 'KF_Hidden_Trend_Norm', 'KF_VWAP_Residual']print('Kalman filter factors defined.')

In [ ]:
# ============================================================# Cell 10: 成交量分布POC距离因子 (wxy_new_factor)# ============================================================@njit(cache=True)def _find_poc_numba(typical_prices, volumes, window, bins):    """    Numba加速的POC检测。    在每个滑动窗口内计算成交量直方图，返回成交量最大的价格区间中心。    """    T = len(typical_prices)    poc_array = np.zeros(T, dtype=np.float64)    for t in range(window - 1, T):        t0 = t - window + 1        prices_win = typical_prices[t0:t+1]        vols_win = volumes[t0:t+1]        lo = np.min(prices_win)        hi = np.max(prices_win)        if hi - lo < 1e-8:            poc_array[t] = (lo + hi) / 2            continue        hist = np.zeros(bins, dtype=np.float64)        bin_width = (hi - lo) / bins        for i in range(window):            p = prices_win[i]            v = vols_win[i]            bidx = int((p - lo) / bin_width)            if bidx >= bins:                bidx = bins - 1            elif bidx < 0:                bidx = 0            hist[bidx] += v        max_idx = 0        max_val = hist[0]        for b in range(1, bins):            if hist[b] > max_val:                max_val = hist[b]                max_idx = b        # POC = 最大成交量区间的中心        poc_array[t] = lo + (max_idx + 0.5) * bin_width    return poc_array.astype(np.float32)def generate_poc_factor(O, H, L, C, V, window=60, bins=50):    """    生成成交量分布POC距离因子。    返回: (T,) 的一维数组    """    T = len(C)    typical_prices = (H.astype(np.float64) + L.astype(np.float64) + C.astype(np.float64)) / 3.0    poc = _find_poc_numba(typical_prices, V.astype(np.float64), window, bins)    # ATR for normalization    atr = np.zeros(T, dtype=np.float32)    for t in range(1, T):        tr = max(H[t]-L[t], abs(H[t]-C[t-1]), abs(L[t]-C[t-1]))        atr[t] = tr    atr_ema = bn.move_mean(atr, window=window, min_count=1)    poc_distance = np.where(atr_ema > EPS,        (C - poc) / (atr_ema + EPS), 0.0)    return poc_distance.astype(np.float32)print('POC distance factor defined.')

In [ ]:
# ============================================================# Cell 11: 文献综述因子 (wxy_new_factor)# ============================================================def generate_literature_factors(O, H, L, C, V):    """    生成12个文献综述因子。    返回: (T, 12) 的二维数组    """    T = len(C)    lit = np.zeros((T, 12), dtype=np.float32)    hl_range = H - L    log_C = np.log(np.maximum(C, EPS))    log_ret = np.zeros(T, dtype=np.float32)    log_ret[1:] = log_C[1:] - log_C[:-1]    # 1. Parkinson Volatility    parkinson = np.sqrt(np.log(H / np.maximum(L, EPS))**2 / (4 * np.log(2)))    lit[:, 0] = parkinson    # 2. Rogers-Satchell Volatility    rs_vol = np.sqrt(        np.log(H / np.maximum(C, EPS)) * np.log(H / np.maximum(O, EPS)) +        np.log(L / np.maximum(C, EPS)) * np.log(L / np.maximum(O, EPS))    )    lit[:, 1] = np.nan_to_num(rs_vol, nan=0.0)    # 3. Lower Shadow Fraction    lower_shadow = np.where(hl_range > EPS,        (np.minimum(O, C) - L) / hl_range, 0)    lit[:, 2] = lower_shadow    # 4. Body Fraction    body_frac = np.where(hl_range > EPS,        np.abs(C - O) / hl_range, 0)    lit[:, 3] = body_frac    # 5. Close Position    close_pos = np.where(hl_range > EPS,        (C - L) / hl_range, 0.5)    lit[:, 4] = close_pos    # 6. Roll Spread Proxy    dC = np.diff(C)    dC1 = dC[1:]    dC0 = dC[:-1]    cov_dc = bn.move_mean(dC1 * dC0, window=60, min_count=1)    roll_spread = np.zeros(T, dtype=np.float32)    roll_spread[2:] = np.where(cov_dc < 0,        2 * np.sqrt(np.maximum(-cov_dc, 0)), 0)    lit[:, 5] = roll_spread    # 7. Kaufman Efficiency Ratio    net_change = np.abs(C[60:] - C[:-60])    path_length = bn.move_sum(np.abs(np.diff(C, prepend=C[:1])), window=60, min_count=1)[60:]    kaufman_er = np.zeros(T, dtype=np.float32)    kaufman_er[60:] = np.where(path_length > EPS,        net_change / (path_length + EPS), 0)    lit[:, 6] = kaufman_er    # 8. Donchian Width    hh60 = bn.move_max(H, window=60, min_count=1)    ll60 = bn.move_min(L, window=60, min_count=1)    lit[:, 7] = (hh60 - ll60) / (C + EPS)    # 9. Price-Volume Correlation    dP_small = np.diff(C, prepend=C[:1])    mean_dP = bn.move_mean(dP_small, window=60, min_count=1)    mean_V = bn.move_mean(V, window=60, min_count=1)    cov_PV = bn.move_mean((dP_small - mean_dP) * (V - mean_V), window=60, min_count=1)    std_P = bn.move_std(dP_small, window=60, min_count=1)    std_V = bn.move_std(V, window=60, min_count=1)    lit[:, 8] = np.where((std_P * std_V) > EPS,        cov_PV / (std_P * std_V + EPS), 0)    # 10. Volume Weighted Close    vw_num = bn.move_sum(V * C, window=60, min_count=1)    vw_den = bn.move_sum(V, window=60, min_count=1)    lit[:, 9] = vw_num / (vw_den + EPS)    # 11. Realized Vol 5M    rv5 = np.sqrt(bn.move_sum(log_ret**2, window=5, min_count=1))    lit[:, 10] = rv5    # 12. Risk-Adjusted Momentum    mom60 = np.zeros(T, dtype=np.float32)    mom60[60:] = (C[60:] - C[:-60]) / (C[:-60] + EPS)    std60 = bn.move_std(log_ret, window=60, min_count=1)    lit[:, 11] = np.where(std60 > EPS,        mom60 / (std60 + EPS), 0)    return litLIT_NAMES = [    'Parkinson_Volatility_1M', 'Rogers_Satchell_Vol_1M',    'Lower_Shadow_Frac', 'Body_Fraction', 'Close_Position',    'Roll_Spread_Proxy_1M', 'Kaufman_ER_60M', 'Donchian_Width_60M',    'Price_Volume_Corr_60M', 'Volume_Weighted_Close_60M',    'RV_5M_Realized_Vol', 'Risk_Adj_Momentum_60M']print('Literature review factors defined.')

In [ ]:
# ============================================================# Cell 12: IC评估函数# ============================================================def calculate_pearson_ic(factors, label):    """    计算因子的Pearson IC。    factors: (T, n_factors) 的二维数组    label: (T,) 的一维数组 (未来收益率)    返回: (n_factors,) 的一维IC数组    """    # 过滤NaN标签    vm = ~np.isnan(label)    y = label[vm]    X = factors[vm, :]    # 处理NaN因子值 (置零)    nm = np.isnan(X)    X = np.where(nm, 0, X)    # 有效样本计数    vc = (~nm).sum(axis=0).astype(np.float64)    vc = np.where(vc == 0, 1e-8, vc)    # Pearson相关系数    ym = np.mean(y)    xm = X.sum(axis=0) / vc    yc = y - ym    xc = np.where(nm, 0, X - xm)    cov = np.dot(yc, xc) / vc    ys = np.sqrt(np.mean(yc ** 2))    xs = np.sqrt(np.sum(xc ** 2, axis=0) / vc)    ic = cov / (ys * xs + EPS)    # 样本覆盖率不足10%的因子IC置零    ic = np.where(vc < len(y) * 0.1, 0, ic)    return icdef evaluate_factors_ic(factors_matrix, ret5, ret60, factor_names):    """    评估一组因子的IC表现。    factors_matrix: (T, n) 的二维数组    ret5, ret60: (T,) 的未来收益率    factor_names: list of str    返回: 字典列表 (每个因子一行)    """    ic_ret5 = calculate_pearson_ic(factors_matrix, ret5)    ic_ret60 = calculate_pearson_ic(factors_matrix, ret60)    results = []    for i, name in enumerate(factor_names):        mean5 = ic_ret5[i]        mean60 = ic_ret60[i]        ir5 = abs(mean5)        ir60 = abs(mean60)        score5 = abs(mean5) * ir5        score60 = abs(mean60) * ir60        results.append({            'Factor_Name': name,            'Ret5_Mean_IC': mean5,            'Ret60_Mean_IC': mean60,            'Ret5_Score': score5,            'Ret60_Score': score60,            'Comprehensive_Score': score5 + score60        })    return resultsprint('IC evaluation functions defined.')

In [ ]:
# ============================================================# Cell 13: 整合所有因子 — 一键生成函数# ============================================================def generate_all_factors(O, H, L, C, V):    """    整合所有因子的生成函数，返回因子矩阵和名称列表。    O, H, L, C, V: 各为长度T的一维numpy数组    返回: (factor_matrix, factor_names)    """    T = len(C)    all_factors = []    all_names = []    # 1. A-H 八大类优化因子 (25个)    f1 = generate_optimized_factors(O, H, L, C, V)    all_factors.append(f1)    all_names.extend(OPTIMIZED_FACTOR_NAMES)    # 2. Signed Power Round 1 (35个)    f2 = generate_sp_round1_factors(O, H, L, C, V)    all_factors.append(f2)    all_names.extend(SP_ROUND1_NAMES)    # 3. Signed Power Round 2 (35个) [可选: 速度较慢]    # f3 = generate_sp_round2_factors(O, H, L, C, V)    # all_factors.append(f3)    # all_names.extend(SP_ROUND2_NAMES)    # 4. 分数阶差分因子 (6个)    f4 = generate_frac_diff_factors(O, H, L, C, V)    all_factors.append(f4)    all_names.extend(FRAC_DIFF_NAMES)    # 5. 卡尔曼滤波因子 (3个)    f5 = generate_kalman_factors(O, H, L, C, V)    all_factors.append(f5)    all_names.extend(KF_NAMES)    # 6. POC距离因子 (1个)    f6 = generate_poc_factor(O, H, L, C, V).reshape(-1, 1)    all_factors.append(f6)    all_names.append('Distance_to_POC_60M')    # 7. 文献综述因子 (12个)    f7 = generate_literature_factors(O, H, L, C, V)    all_factors.append(f7)    all_names.extend(LIT_NAMES)    # 拼接所有因子    factor_matrix = np.concatenate(all_factors, axis=1)    return factor_matrix, all_namesprint(f'generate_all_factors: Ready to produce factor_matrix and names')

In [ ]:
# ============================================================# Cell 14: 使用示例 — 在数据集上运行# ============================================================# 示例：加载单个数据集并计算所有因子## data = np.load('dataset0_train_ohlcv.npy')  # shape (N, 8): [idx, O, H, L, C, V, ret5, ret60]# O, H, L, C, V = data[:,1], data[:,2], data[:,3], data[:,4], data[:,5]# ret5, ret60 = data[:,6], data[:,7]## # 生成所有因子# factor_matrix, factor_names = generate_all_factors(O, H, L, C, V)## # 评估IC# results = evaluate_factors_ic(factor_matrix, ret5, ret60, factor_names)## # 按综合得分排序# sorted_results = sorted(results, key=lambda x: x['Comprehensive_Score'], reverse=True)## # 打印Top 10# print('Top 10 Factors by Comprehensive Score:')# for i, r in enumerate(sorted_results[:10]):#     print(f'{i+1}. {r["Factor_Name"]}  |  Ret5 IC={r["Ret5_Mean_IC"]:.4f}  |  Ret60 IC={r["Ret60_Mean_IC"]:.4f}  |  Score={r["Comprehensive_Score"]:.6f}')print('Usage example ready. See comments above for usage instructions.')print('To run on actual data, load .npy files and call generate_all_factors().')

---# 第五部分：总结与展望## 主要成果1. **构建了117个量化因子**，覆盖8大优化方法 + 有符号幂变换 + 分数阶差分 + 卡尔曼滤波 + POC距离 + 文献因子2. **系统性地评估了所有因子的IC表现**，结果汇总在 `Consolidated_Factor_IC_Summary.csv`3. **发现了有符号幂变换 (Signed Power Transformation)** 作为最有效的优化方法，特别是 p=2.0 的平方变换4. **EMA收益偏离 (ema_ret)** 被确认为最稳定的基础信号，经过有符号幂变换后IC表现最优## 关键性能指标| 因子来源 | 因子数 | 最佳因子 | 最佳综合得分 ||---------|--------|---------|------------|| 基础优化 (A-H) | 25 | F3_Delta5_LogHL_Range | 0.0124 || Signed Power Round 1 | 35 | SP20_ema_ret_60M | 0.1359 || Signed Power Round 2 | 35 | SP20_ema_ret_15M | 0.1138 || 分数阶差分 | 6 | FracDiff_LogPrice_60M | — || 卡尔曼滤波 | 3 | KF_FairValue_Dev | — || POC距离 | 1 | Distance_to_POC_60M | — || 文献因子 | 12 | Roll_Spread_Proxy_1M | 0.0260 |## 建议后续方向1. **集成学习**: 将Top因子通过GBDT/神经网络进行非线性组合2. **多周期验证**: 在更多数据集和时间周期上验证因子稳定性3. **实盘模拟**: 选择综合得分>0.05的因子进行样本外交易模拟4. **因子正交化**: 对高IC因子进行正交化处理，减少共线性